# 实验 1：认识 Qwen3-0.6B 的外部骨架

## 今天只回答一个问题

**一个 token 进入 Qwen3-0.6B 之后，要依次经过哪些主要模块？**

本实验只观察模型的**顶层流水线**和**模块树**：

1. 先看一张总览图，建立整体地图；
2. 再从真实模型对象自动生成一棵精简树；
3. 最后把图和树放在一起，确认它们描述的是同一副骨架。

今天暂时不打开 Transformer Block 的内部。Attention、MLP、GQA 和权重矩阵会在后续实验中分别学习。模型会加载约 1.2GB 权重，但本实验不生成文本、不训练模型。

In [1]:
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

from qwen_kit import MODEL_PATH, show, show_path, observe, setup

show([('torch', torch.__version__),
      ('transformers', transformers.__version__),
      ('模型路径', show_path(MODEL_PATH))],
     header=['环境', '值'])

环境,值
torch,2.13.0+cu130
transformers,5.15.1
模型路径,models/Qwen3-0.6B-Base


In [2]:
# 只加载模型，不做生成；设备固定 CPU，dtype 用 bf16 —— 本实验只看形状，精度无关。
DEVICE = 'cpu'

# 全程关闭梯度。只研究推理，不训练。
torch.set_grad_enabled(False)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    dtype=torch.bfloat16,
).to(DEVICE).eval()

config = model.config
N_LAYERS = config.num_hidden_layers
HIDDEN = config.hidden_size
VOCAB = config.vocab_size

show([('模型类', type(model).__name__, ''),
      ('device', str(next(model.parameters()).device), ''),
      ('dtype', str(next(model.parameters()).dtype), ''),
      ('已切到 eval（training=False）', not model.training, '✓ 表示确认通过'),
      ('已关闭梯度', not torch.is_grad_enabled(), ''),
      ('层数 / 宽度 / 词表', f'{N_LAYERS} / {HIDDEN} / {VOCAB}', '全部现推自 config'),
      ('参数量', f'{sum(p.numel() for p in model.parameters()):,}', '')],
     header=['加载结果', '值', '说明'])

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

加载结果,值,说明
模型类,Qwen3ForCausalLM,
device,cpu,
dtype,torch.bfloat16,
已切到 eval（training=False）,✓,✓ 表示确认通过
已关闭梯度,✓,
层数 / 宽度 / 词表,28 / 1024 / 151936,全部现推自 config
参数量,"596,049,920",


## 1. 先看模型的总览图

今天只观察模型的**外部流水线**，先不打开 Transformer Block 的内部。

<div align="center">
  <img src="../assets/qwen3-backbone-overview.png" alt="Qwen3 顶层结构总览图" width="360" style="max-width: 100%; border: 1px solid #cbd5e1; border-radius: 8px;">
</div>

先把 Block 当作黑盒：它接收 hidden states，处理后交给下一阶段。Attention、MLP、GQA 和权重矩阵会在后续实验中分别学习。

### 图的数值版本

图上那五个阶段，每一步真实的输出形状是多少？跑一次 forward 就能看到。

In [3]:
PROMPT = '北京是中国的首都，巴黎是法国的'
input_ids = tokenizer(PROMPT, return_tensors='pt').input_ids.to(DEVICE)

setup(model, tokenizer)
qwen = observe(model, input_ids)

# 五个顶层阶段的形状，全部取自官方模型的 hook。
_skeleton = [
    ('input_ids', input_ids, 'B=1, S=9 的整数张量'),
    ('embed_tokens', qwen.embed, '查表：行号 → 1024 维向量'),
    ('layers.0', qwen.L[0].out, '保形'),
    ('layers.1', qwen.L[1].out, '保形  ⋯ 中间 26 层省略'),
    (f'layers.{N_LAYERS - 1}', qwen.L[N_LAYERS - 1].out, '保形'),
    ('final_norm', qwen.final_norm, 'RMSNorm，不改形状'),
    ('lm_head', qwen.logits, f'1024 → {VOCAB} 词表打分'),
]
show([(n, str(tuple(t.shape)), w) for n, t, w in _skeleton],
     header=['阶段', '输出形状', '说明'])

[transformers] `sdpa` attention does not support `output_attentions=True`. Please set your attention to `eager` if you want any of these features.


阶段,输出形状,说明
input_ids,"(1, 9)","B=1, S=9 的整数张量"
embed_tokens,"(1, 9, 1024)",查表：行号 → 1024 维向量
layers.0,"(1, 9, 1024)",保形
layers.1,"(1, 9, 1024)",保形 ⋯ 中间 26 层省略
layers.27,"(1, 9, 1024)",保形
final_norm,"(1, 9, 1024)",RMSNorm，不改形状
lm_head,"(1, 9, 151936)",1024 → 151936 词表打分


## 2. 从真实模型对象生成精简模块树

下面的树不是手写示意图。代码会读取刚刚加载的 `model` 对象，自动取得模块类名、层数和维度。

阅读缩进的方法：

- `model` 和 `lm_head` 是 `Qwen3ForCausalLM` 直接包含的两个顶层部分；
- `embed_tokens`、`layers`、`norm` 都位于内部的 `Qwen3Model` 中；
- `layers × 28` 暂时作为一组黑盒，不在本实验继续展开。

In [4]:
backbone = model.model
embedding = backbone.embed_tokens
blocks = backbone.layers
final_norm = backbone.norm
lm_head = model.lm_head

print(type(model).__name__)
print(f"├── model: {type(backbone).__name__}")
print(
    f"│   ├── embed_tokens: {type(embedding).__name__}"
    f"({embedding.num_embeddings:,}, {embedding.embedding_dim})"
)
print(
    f"│   ├── layers: {type(blocks).__name__} × {len(blocks)} "
    f"[{type(blocks[0]).__name__}]"
)
print(f"│   └── norm: {type(final_norm).__name__}({config.hidden_size})")
print(
    f"└── lm_head: {type(lm_head).__name__}"
    f"({lm_head.in_features}, {lm_head.out_features}, bias={lm_head.bias is not None})"
)


Qwen3ForCausalLM
├── model: Qwen3Model
│   ├── embed_tokens: Embedding(151,936, 1024)
│   ├── layers: ModuleList × 28 [Qwen3DecoderLayer]
│   └── norm: Qwen3RMSNorm(1024)
└── lm_head: Linear(1024, 151936, bias=False)


## 3. 小结：今天只记住这副外部骨架

```text
Input Token
    ↓
Embedding                model.model.embed_tokens
    ↓
28 × Transformer Block   model.model.layers
    ↓
Final RMSNorm            model.model.norm
    ↓
LM Head                  model.lm_head
    ↓
Output Logits
```

完成本实验后，你应该能够回答：

1. 一段 token 序列进入模型后，会依次经过哪些顶层阶段？
2. `Qwen3ForCausalLM`、`Qwen3Model`、`embed_tokens`、`layers`、`norm` 和 `lm_head` 在模块树中是什么包含关系？

本实验到这里停止。下一实验再选择一个黑盒打开观察。